# Prétraitement des données

Démonstration du pipeline défini dans `src/preprocessing.py` :
chargement, standardisation, split stratifié et SMOTE.


In [1]:
import sys
import os

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('../src'))
from preprocessing import (
    load_data,
    scale_features,
    split_data,
    balance_data,
    prepare_full_pipeline,
)


## 1. Chargement


In [2]:
df = load_data('../data/creditcard.csv')
df.head()


Données chargées : 284807 transactions, 31 colonnes.


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2. Standardisation de Time et Amount


In [3]:
df_scaled, scaler = scale_features(df, fit=True)
df_scaled[['Time', 'Amount']].describe()


,Time,Amount
count,2.848070e+05,2.848070e+05
mean,-5.109395e-17,-3.672378e-17
std,1.000002e+00,1.000002e+00
min,-1.996583e+00,-3.532294e-01
25%,-8.552120e-01,-3.308401e-01
50%,-2.131453e-01,-2.652715e-01
75%,9.372174e-01,-4.471707e-02
max,1.642058e+00,1.023622e+02


`Time` et `Amount` sont maintenant centrées sur 0 et d'écart-type 1.
Les colonnes V1-V28 ne sont pas touchées.


## 3. Split stratifié


In [4]:
X_train, X_test, y_train, y_test = split_data(df_scaled)
print(f"Proportion de fraudes train : {y_train.mean() * 100:.3f} %")
print(f"Proportion de fraudes test  : {y_test.mean() * 100:.3f} %")


Split : 227845 en entraînement, 56962 en test.
Proportion de fraudes train : 0.173 %
Proportion de fraudes test  : 0.172 %


## 4. Rééquilibrage SMOTE


In [5]:
X_res, y_res = balance_data(X_train, y_train)
print(f"Distribution y après SMOTE :\n{y_res.value_counts()}")


Rééquilibrage SMOTE : 227845 -> 454902 exemples. Fraudes : 394 -> 227451.
Distribution y après SMOTE :
Class
0    227451
1    227451
Name: count, dtype: int64


## 5. Pipeline complet en une fonction


In [6]:
data = prepare_full_pipeline(
    path='../data/creditcard.csv',
    test_size=0.2,
    use_smote=True,
    scaler_path='../models/scaler.joblib',
)
print({k: v.shape if hasattr(v, 'shape') else type(v).__name__ for k, v in data.items()})


Données chargées : 284807 transactions, 31 colonnes.
Scaler sauvegardé dans : ../models/scaler.joblib
Split : 227845 en entraînement, 56962 en test.


Rééquilibrage SMOTE : 227845 -> 454902 exemples. Fraudes : 394 -> 227451.
{'X_train': (454902, 30), 'X_test': (56962, 30), 'y_train': (454902,), 'y_test': (56962,), 'scaler': 'StandardScaler'}


## Conclusion

Le pipeline est **réutilisable** (modules `src/`) et reproductible
(random_state fixé). Le scaler ajusté est sauvegardé pour être
réutilisé en prédiction.
